In [24]:
from pathlib import Path
import re
import os

input_dir = Path("/mnt/nfs_dev/zah/data/book")
files = sorted([f for f in input_dir.rglob("*.md") if 'debug' not in f.parts])
print(files)

[PosixPath('/mnt/nfs_dev/zah/data/book/150种人造板材配方与制作 (李东光主编).pdf/150种人造板材配方与制作 (李东光主编).pdf.md'), PosixPath('/mnt/nfs_dev/zah/data/book/2011年西门子自动化专家会议论文集 上 (西门子（中国）有限公司编) .pdf/2011年西门子自动化专家会议论文集 上 (西门子（中国）有限公司编) .pdf.md'), PosixPath('/mnt/nfs_dev/zah/data/book/2011年西门子自动化专家会议论文集 下 (西门子（中国）有限公司编) .pdf/2011年西门子自动化专家会议论文集 下 (西门子（中国）有限公司编) .pdf.md'), PosixPath('/mnt/nfs_dev/zah/data/book/AI加速器架构设计与实现. (甄建勇).pdf/AI加速器架构设计与实现. (甄建勇).pdf.md'), PosixPath('/mnt/nfs_dev/zah/data/book/AI系统 原理与架构 (ZOMI酱, 陈仲铭, 苏统华) .pdf/AI系统 原理与架构 (ZOMI酱, 陈仲铭, 苏统华) .pdf.md'), PosixPath('/mnt/nfs_dev/zah/data/book/Accelerated Testing Statistical Models, Test Plans, and Data Analysis (Wiley Series in Probability and Statistics) (Wayne B. Nelson) .pdf/Accelerated Testing Statistical Models, Test Plans, and Data Analysis (Wiley Series in Probability and Statistics) (Wayne B. Nelson) .pdf.md'), PosixPath('/mnt/nfs_dev/zah/data/book/BGP design and implementation (Zhang, Randy, author, Bartell, Micah, author).pdf/BGP desi

In [25]:
def has_chinese_char(text):
    """
    检测字符串中是否包含中文字符
    原理：检查字符的 Unicode 编码是否在常用汉字范围内
    """
    for char in text:
        if '\u4e00' <= char <= '\u9fff':
            return True
    return False

def is_english_file(filename):
    """
    判断文件是否为英文文件
    规则：文件名中不存在任何中文字符，即为英文文件
    """
    # 1. 获取文件名主体（去除后缀）
    # 兼容处理：如果输入是字符串路径，先转为 Path 对象
    if isinstance(filename, str):
        p = Path(filename)
    else:
        p = filename
    
    name_without_ext = p.stem
    
    # 2. 核心逻辑：如果没有中文字符，就认为是英文文件
    # 如果检测函数返回 False (无中文)，则 is_english 为 True
    return not has_chinese_char(name_without_ext)

In [26]:
def extract_toc_robust(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    start_idx = -1
    for i, line in enumerate(lines):
        clean_line = line.strip().replace(" ", "")
        if re.search(r'^#*目录$', clean_line) or re.search(r'^#*CONTENTS$', clean_line, re.I) or re.search(r'^#*目 录$', clean_line):
            start_idx = i
            break
            
    if start_idx == -1: return "False", None, None, file_path

    # --- 阶段 1: 采样前 5-8 行有效标题作为锚点集合 ---
    anchor_set = []
    anchor_clean_set = []
    scan_limit = 8 # 稍微多扫几行以确保抓到核心标题
    found_count = 0
    
    for i in range(start_idx + 1, int(len(lines)*0.2)):
        line_content = lines[i].strip().replace("#", "").strip()
        if not line_content: continue

        pure_title = re.sub(r'\s+\d+$', '', line_content).strip()
        anchor_clean_set.append(pure_title.replace(" ", "") )
        
        # if pure_title in anchor_set or pure_title in anchor_clean_set:
        #     return anchor_set
        
        anchor_set.append(pure_title)
        found_count += 1
            
        if found_count >= scan_limit or i > start_idx + 20: # 找到5个或扫描超过20行就停止
            break
    # anchor_clean_set =  [item.replace(" ", "") for item in anchor_set]
    end_start_idx = i + 1

    # --- 阶段 2: 寻找终点 ---
    end_idx = -1
    i = end_start_idx
    limit = int(len(lines) * 0.2)
    
    while i < limit:
        line_raw = lines[i].strip()
        clean_content = line_raw.replace("#", "").strip().replace(" ", "")
        
        # 1. 检查是否匹配锚点集合 (正文标题重现)
        is_anchor_match = clean_content and any(clean_content.lower() in anchor.lower() for anchor in anchor_set)
        is_anchor_clean_match = clean_content and any(clean_content.lower() in anchor.lower() for anchor in anchor_clean_set)
        is_match = any(clean_content == anchor.replace(" ", "") for anchor in anchor_set)
        has_page_num = re.search(r'\s\d+$', line_raw)
    
        if (is_anchor_match and not has_page_num) or(is_anchor_clean_match and not has_page_num) or  is_match:
            end_idx = i
            break
    
        # 2. 兜底逻辑：贪婪匹配参考文献
        # 匹配规则：去掉符号、数字、标点后，内容等于“参考文献”等
        # 使用 re.sub 去掉非中文字符和非英文字母
        simplified_content = re.sub(r'[^\u4e00-\u9fa5a-zA-Z]', '', clean_content)
        is_ref_header = re.match(r'^(参考?文献|Bibliography|References)$', simplified_content, re.IGNORECASE)
    
        if is_ref_header:
            # 初始标记为当前行
            end_idx = i + 1
            
            # 开启贪婪探测模式：向后查找 50 行
            look_ahead_idx = i + 1
            while look_ahead_idx < min(i + 51, limit):
                next_line = lines[look_ahead_idx].strip()
                next_clean = next_line.replace("#", "").strip().replace(" ", "")
                next_simplified = re.sub(r'[^\u4e00-\u9fa5a-zA-Z]', '', next_clean)
                
                # 如果在 50 行内又发现了一个独立的“参考文献”行
                if re.match(r'^(参考?文献|Bibliography|References)$', next_simplified, re.IGNORECASE):
                    end_idx = look_ahead_idx + 1
                    # 重置 i 到新发现的位置，以便外层循环能继续从这里开始下一次 50 行的探测
                    i = look_ahead_idx 
                    # 更新 look_ahead_idx 重新开始算 50 行
                    look_ahead_idx = i + 1
                    continue
                
                look_ahead_idx += 1
            
            # 探测结束，跳出外层循环
            break
        
        i += 1

    # --- 阶段 3: 返回截取结果 ---
    if end_idx != -1:
        # 如果是分行标题，end_idx 可能是标题的第二行，视情况可以向上调1行
        return lines[start_idx:end_idx], start_idx, end_idx, file_path
    else:
        # 极端情况兜底：取目录开始后的300行
        return "False", None, None
    

In [36]:
def auto_detect_hierarchy(md_content):
    # 1. 预定义可能的特征提取正则 (按优先级排序)
    feature_patterns = [
        r'第[一二三四五六七八九十\d]+篇',
        r'第[一二三四五六七八九十\d]+章',
        r'第[一二三四五六七八九十\d]+节',
        r'第[一二三四五六七八九十\d]+单元',
        r'第[一二三四五六七八九十\d]+部分',
        
        r'(?:单元|任务|项目)[一二三四五六七八九十\d]+', # 单元、任务

        r'\d+\.\d+\.\d+', # 1.1.1
        r'\d+\.\d+',      # 1.1
        r'\d+[\.．]\s*',       # 1. (带点),后面有无空格都可以，全角半角都支持
        r'^[一二三四五六七八九十]+、', # 一、
        r'^\d+\s',        # 1 (空格)
        r'^\(\d+\)',      # (1)
        r'^\([一二三四五六七八九十]+\)' # (一)
    ]

    level_queue = [] # 存储本篇文档发现的特征指纹
    output_lines = []
    
    for line in md_content:
        raw_content = line.strip()
        raw_content = raw_content.lstrip('#')
        raw_content = raw_content.strip()
        if not raw_content: continue
        
        # 移除行首旧的 # 和行尾页码
        clean_text = re.sub(r'^#+\s*', '', raw_content)
        clean_text = re.sub(r'[\.\…]{2,}', '', clean_text)
        pattern = r'\s*\(\d+\)\s*$|\s+\d+$'
        clean_text = re.sub(pattern, '', clean_text, flags=re.MULTILINE)
        
        current_fingerprint = None
        
        # 匹配特征
        for p in feature_patterns:
            match = re.search(p, clean_text)
            if match:
                # 修改部分：先标准化分隔符，再提取特征
                # 将匹配到的文本中的 '.' 或 '、' 及其后的空格统一为 '. '
                normalized_match = re.sub(r'[\.、]\s*', '. ', match.group())
                
                # 提取纯特征，忽略具体数字
                current_fingerprint = re.sub(r'\d+', '\\\\d', normalized_match)
                current_fingerprint = re.sub(r'[一二三四五六七八九十]+', 'CN', current_fingerprint)
                break
        
        if current_fingerprint:
            # 如果是新特征，加入层级队列
            if current_fingerprint not in level_queue:
                level_queue.append(current_fingerprint)
            
            # 计算当前层级 (Index 从 0 开始，所以 +1)
            depth = level_queue.index(current_fingerprint) + 1
            output_lines.append(f"{'#' * depth} {clean_text}")
        else:
            # # 如果没匹配到任何特征，视为普通文本或维持原样
            # if clean_text == "参考文献" or clean_text.lower() == "bibliography" or clean_text.lower() == "references":
            #     output_lines.append(f"{'#' * 1} {clean_text}")
            # else:
            if raw_content.startswith("#"):
                output_lines.append(raw_content)
            
    return output_lines

In [37]:
false_md = []
toc_list = []
for i in range(len(files)):
    if not (is_english_file(files[i])):
        toc_blocks, start_idx, end_idx, file_path = extract_toc_robust(files[i])
        if toc_blocks == "False":
            false_md.append(files[i])
        else:
            if toc_blocks == "False":
                false_md.append(files[i]) 
            else:
                toc_blocks = [block for block in toc_blocks if len(block) <= 50]
                print(f"------------------------{i}--------------------------")
                print(toc_blocks)
                toc_list.append((toc_blocks, start_idx, end_idx, file_path))
    else:
        pass
        # TODO：处理英文
print(false_md)


------------------------3--------------------------
['# CONTENTS\n', '\n', '# 目 录\n', '\n', '# 前言\n', '\n', '# 第1章 卷积神经网络\n', '\n', '1.1 神经网络的结构· 2\n', '\n', '1.2 GCN 4\n', '\n', '1.3 网络的基本块 7\n', '\n', '1.4 网络的算子 17\n', '\n', '1.5 网络参数量与运算量 ..... 29\n', '\n', '1.6 加速器编程模型 31\n', '\n', '1.7 硬件加速器架构分类 33\n', '\n', '# 第2章 运算子系统的设计 35\n', '\n', '2.1 数据流设计 35\n', '\n', '2.2 算力与带宽 38\n', '\n', '2.2.1 算力与输入带宽 38\n', '\n', '2.2.2 算力与输出带宽 41\n', '\n', '2.3 卷积乘法阵列 43\n', '\n', '2.3.1 Conv算法详解 43\n', '\n', '2.3.2NVDLA的乘法阵列 47\n', '\n', '2.3.3 TPU的乘法阵列 59\n', '\n', '2.3.4 GPU的乘法阵列 66\n', '\n', '2.3.5 华为DaVinci的乘法阵列 74\n', '\n', '2.4卷积运算顺序的选择· 80\n', '\n', '2.5 池化模块的设计 81\n', '\n', '# 第3章 存储子系统的设计 86\n', '\n', '3.1 存储子系统概述 86\n', '\n', '3.1.1 存储子系统的组成 86\n', '\n', '3.1.2 内部缓存的设计 89\n', '\n', '3.2 数据格式的定义 97\n', '\n', '3.2.1 特征图的格式 98\n', '\n', '3.2.2 权重的格式 100\n', '\n', '# 第4章 架构优化技术 106\n', '\n', '4.1 运算精度的选择 106\n', '\n', '4.1.1 dynamic fixed point 类型 ..... 109\n', '\n', '4.1.2 bfloat16 类型 .....

In [38]:
processed_toc_list = []
for idx, toc in enumerate(toc_list):
    toc_content, start_idx, end_idx, file_path = toc
    processed_toc = auto_detect_hierarchy(toc_content)
    processed_toc_list.append((processed_toc, start_idx, end_idx, file_path))
# for idx, item in enumerate(processed_toc_list):
#     processwd_toc, start_idx, end_idx, file_path = item
#     rewrite_md(processwd_toc, start_idx, end_idx, file_path)

In [58]:
i = 0
processed_toc, start_idx, end_idx, file_path = processed_toc_list[i]

In [59]:
with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

In [60]:
type(lines)
print(start_idx, end_idx)

152 294


In [61]:
lines = lines[end_idx:]
lines

['# 第1章\n',
 '\n',
 '# 卷积神经网络\n',
 '\n',
 '从2012年AlexNet的出现到2016年火爆全球的AlphaGo围棋大战，人工智能（Artificial Intelligence，AI）一词逐渐被人们熟知。从专业的角度来看，人工智能是一个很宽泛的词汇，大家常说的人工智能一般指的是基于深度学习的人工智能，如图1-1所示，深度学习只是人工智能的一个分支。\n',
 '\n',
 '![](figures/011-20260211155048023502-whxy_956.7301_1463.4730_0.2699_0.5270.jpg)\n',
 '\n',
 '图1-1 深度学习与人工智能\n',
 '\n',
 '深度学习之所以能照进现实，并且在很多领域得到广泛应用，数据、算法、算力三者缺一不可，而本书讨论的重点就是基于深度学习的加速器架构设计，尤其是基于卷积神经网络的加速器设计，以缓解深度学习面临的算力问题。\n',
 '\n',
 '# 1.1 神经网络的结构\n',
 '\n',
 '经过几年的发展，神经网络的应用领域已经非常广泛。没有一种硬件架构能够支持所有的神经网络，从能效比角度来看也是如此，因为不同领域的神经网络的结构可能差异较大，所以设计硬件加速器架构时应该对某个或者某类应用领域有所侧重。当然，也不应该走向另外一个极端，即仅支持某个特定的神经网络。\n',
 '\n',
 '神经网络算法日新月异，演进迅速。专用性过强会大大增加项目风险，虽然硬件架构支持所应用领域的主流神经网络，但也要对算法留有一定的演进空间。这需要对神经网络进行抽象和总结，找出共性并实现。对神经网络进行抽象和总结，可从以下几个方向入手。\n',
 '\n',
 '□浏览目标领域及其邻域的神经网络，了解其功能、结构、异于其他网络的关键点，对神经网络进行分类。\n',
 '\n',
 '□对于目标领域的神经网络进行进一步分析，掌握其网络结构并分类总结，了解数据流向，对整体所需算力、参数量进行统计，对带宽需求做到心中有数。\n',
 '\n',
 '□对于目标领域的神经网络，找到不同网络中重复出现的基本块，抽取其计算过程、数据流向等信息。\n',
 '\n',
 '□ 对于目标网络中的基本块进一步分析，对具体运算细节进行归纳。\n',
 '\n',


In [62]:
processed_toc

['# 第1章 卷积神经网络',
 '## 1.1 神经网络的结构·',
 '## 1.2 GCN',
 '## 1.3 网络的基本块',
 '## 1.4 网络的算子',
 '## 1.5 网络参数量与运算量',
 '## 1.6 加速器编程模型',
 '## 1.7 硬件加速器架构分类',
 '# 第2章 运算子系统的设计',
 '## 2.1 数据流设计',
 '## 2.2 算力与带宽',
 '### 2.2.1 算力与输入带宽',
 '### 2.2.2 算力与输出带宽',
 '## 2.3 卷积乘法阵列',
 '### 2.3.1 Conv算法详解',
 '### 2.3.2NVDLA的乘法阵列',
 '### 2.3.3 TPU的乘法阵列',
 '### 2.3.4 GPU的乘法阵列',
 '### 2.3.5 华为DaVinci的乘法阵列',
 '## 2.4卷积运算顺序的选择·',
 '## 2.5 池化模块的设计',
 '# 第3章 存储子系统的设计',
 '## 3.1 存储子系统概述',
 '### 3.1.1 存储子系统的组成',
 '### 3.1.2 内部缓存的设计',
 '## 3.2 数据格式的定义',
 '### 3.2.1 特征图的格式',
 '### 3.2.2 权重的格式',
 '# 第4章 架构优化技术',
 '## 4.1 运算精度的选择',
 '### 4.1.1 dynamic fixed point 类型',
 '### 4.1.2 bfloat16 类型',
 '## 4.2 硬件资源的复用 …',
 '### 4.2.1 FC',
 '### 4.2.2 de-Conv',
 '### 4.2.3 dilate Conv',
 '### 4.2.4 group Conv',
 '### 4.2.5 3D Conv',
 '### 4.2.6 TC Conv',
 '### 4.2.7 3D Pool',
 '### 4.2.8 Up Sample Pooling',
 '### 4.2.9 多个加速器的级联',
 '## 4.3 Winograd算法和FFT算法',
 '### 4.3.1 Winograd算法解析',
 '### 4.3.2 FFT算法解析',
 '## 4.4 除法变乘法',
 '## 4.

In [63]:
type(processed_toc)

list